# Agentic RFT: Tool Calling

**Role:** training notebook | **Level:** advanced / preview-dependent | **Default mode:** local validation only

Build an Agentic Reinforcement Fine-Tuning (RFT) experiment in which a model must call `search_catalog`, consume the tool result, and return the correct SKU. You will inspect immutable train/validation splits, execute the tool contract locally, calibrate a composite grader, define rollout termination and failure behavior, and construct (but not submit) the fine-tuning payload.

> This notebook performs no uploads, paid training, deployments, or remote calls when run as committed. Service APIs and supported model names can change during preview.

## Learning objectives and prerequisites

You should already understand core RFT, held-out evaluation, and grader calibration. By the end you can explain:

- why SFT imitates a demonstrated next token/tool call while RFT optimizes sequence-level reward;
- how a JSON Schema becomes a hosted tool contract during rollouts;
- how authentication, retries, timeouts, and malformed outputs affect a rollout;
- how exact and fuzzy components create partial credit without rewarding policy-breaking behavior;
- why `max_episode_steps`, terminal answers, and tool failures must end deterministically.

In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
COOKBOOKS_ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "tools"))
sys.path.insert(0, str(COOKBOOKS_ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=False)

from cookbook_utils import (
    configure_logging,
    env_flag,
    evaluate_sku_answers,
    grade_sku_answer,
    load_jsonl,
    require_env,
    response_function_calls,
    responses_input,
    response_output_text,
    responses_tools,
    sha256_file,
    validate_tool_calling_rows,
)
from catalog_server import execute_search_catalog
from shared import (
    create_or_reuse_fine_tuning_job,
    create_project_context,
    load_foundry_config,
    monitor_fine_tuning_job,
    retry_call,
    upload_or_reuse_file,
)
from shared.deployment import ensure_deployment, invoke_response

configure_logging()
TRAIN_PATH = ROOT / "data" / "tool-calling-train.jsonl"
VALIDATION_PATH = ROOT / "data" / "tool-calling-validation.jsonl"
MANIFEST_PATH = ROOT / "data" / "manifest.json"

RUN_REMOTE_TOOL_CHECK = env_flag("FOUNDRY_RUN_REMOTE_TOOL_CHECK")
RUN_PAID_JOBS = env_flag("FOUNDRY_RUN_PAID_JOBS")
RUN_LIVE_EVALUATION = env_flag("FOUNDRY_RUN_LIVE_EVALUATION")
FILE_TIMEOUT_SECONDS = int(os.getenv("FOUNDRY_FILE_TIMEOUT_SECONDS", "600") or "600")
JOB_TIMEOUT_SECONDS = int(os.getenv("FOUNDRY_JOB_TIMEOUT_SECONDS", "14400") or "14400")
POLL_INTERVAL_SECONDS = int(os.getenv("FOUNDRY_POLL_INTERVAL_SECONDS", "60") or "60")
EVALUATION_LIMIT = int(os.getenv("FOUNDRY_EVALUATION_LIMIT", "10") or "10")
print({
    "remote_tool_check": RUN_REMOTE_TOOL_CHECK,
    "paid_jobs": RUN_PAID_JOBS,
    "live_evaluation": RUN_LIVE_EVALUATION,
})


## 1. Provenance, split membership, and schema integrity

The two JSONL files are byte-for-byte copies of the source demo. Renaming makes their role clear; it does not alter examples or move records across splits. The manifest is the audit contract.

In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
expected = {entry["path"]: entry for entry in manifest["files"]}
for path in (TRAIN_PATH, VALIDATION_PATH):
    relative = path.relative_to(ROOT).as_posix()
    actual_hash = sha256_file(path)
    assert actual_hash == expected[relative]["sha256"], relative
    print(relative, expected[relative]["records"], actual_hash)

train_rows = load_jsonl(TRAIN_PATH)
validation_rows = load_jsonl(VALIDATION_PATH)
validate_tool_calling_rows(train_rows, "train")
validate_tool_calling_rows(validation_rows, "validation")
assert len(train_rows) == len(validation_rows) == 10

In [ ]:
example = train_rows[0]
tool_schema = example["tools"][0]
print(json.dumps(tool_schema, indent=2))
print("Reference answer:", example["reference_answer"])

## 2. Tool schema and execution contract

The model sees the function name, description, and JSON Schema. The rollout service sends a function-call envelope to the configured HTTPS server. The server must return a `function_call_output` carrying the same `call_id` and a JSON-encoded `output`.

**Authentication:** prefer a secret manager plus a short-lived bearer token or platform-managed function key. Never place a key in the dataset, schema, notebook, URL query string, or committed `.env`.

**Failure behavior:** reject malformed JSON and invalid argument types with a non-success status; use bounded request timeouts; retry only transient transport/5xx/429 failures with backoff; do not retry authentication, schema, or policy errors. A timeout or invalid response should fail that tool step and yield no fabricated observation.

In [ ]:
local_result = execute_search_catalog("Urban Puffer", top_k=1)
assert local_result == {
    "items": [{"sku": "JKT-URB-009", "name": "Urban Puffer", "price": 149.50}]
}
local_result

### Optional remote contract check

Set `FOUNDRY_RUN_REMOTE_TOOL_CHECK=true` only after hosting the provided tool server and setting `FOUNDRY_TOOL_SERVER_URL`. The check uses a bounded HTTP timeout, sends optional authentication from `FOUNDRY_TOOL_AUTH_TOKEN`, verifies the `function_call_output` envelope, and surfaces non-success responses. Offline execution skips the request.


In [ ]:
remote_contract = "skipped"
if RUN_REMOTE_TOOL_CHECK:
    values = require_env("FOUNDRY_TOOL_SERVER_URL")
    import httpx

    headers = {}
    token = os.getenv("FOUNDRY_TOOL_AUTH_TOKEN", "").strip()
    if token:
        headers["X-Functions-Key"] = token

    def check_remote_tool():
        with httpx.Client(timeout=20) as http_client:
            response = http_client.post(
                values["FOUNDRY_TOOL_SERVER_URL"],
                headers=headers,
                json={
                    "call_id": "call_contract_check",
                    "query": "Urban Puffer",
                    "top_k": 1,
                },
            )
            response.raise_for_status()
            payload = response.json()
            if payload.get("type") != "function_call_output" or not payload.get("output"):
                raise RuntimeError("Remote tool returned an invalid function_call_output envelope")
            output = json.loads(payload["output"])
            expected = execute_search_catalog("Urban Puffer", top_k=1)
            if output != expected:
                raise RuntimeError("Remote tool result does not match the local contract")
            return payload

    remote_contract = retry_call(check_remote_tool, operation_name="check catalog tool contract")
remote_contract


## 3. Grader design and calibration

A strict exact-match grader prevents reward for the wrong SKU. A low-weight fuzzy component gives limited credit for harmless formatting differences. Keep `invalid_grade=0`: missing output, grader exceptions, malformed references, or non-numeric results must never look successful. Calibrate on adversarial cases before training.

In [ ]:
calibration = {
    "correct": grade_sku_answer("JKT-URB-009", "JKT-URB-009"),
    "wrong_sku": grade_sku_answer("JKT-ALP-001", "JKT-URB-009"),
    "extra_text": grade_sku_answer("The SKU is JKT-URB-009", "JKT-URB-009"),
    "empty": grade_sku_answer("", "JKT-URB-009"),
}
calibration

In [ ]:
multi_grader = {
    "type": "multi",
    "name": "strict_partial_credit",
    "graders": {
        "exact": {
            "type": "string_check",
            "name": "exact",
            "operation": "eq",
            "input": "{{sample.output_text}}",
            "reference": "{{item.reference_answer}}",
        },
        "fuzzy": {
            "type": "text_similarity",
            "name": "fuzzy",
            "evaluation_metric": "fuzzy_match",
            "input": "{{sample.output_text}}",
            "reference": "{{item.reference_answer}}",
        },
    },
    "calculate_output": "0.9 * exact + 0.1 * fuzzy",
}
assert multi_grader["calculate_output"] == "0.9 * exact + 0.1 * fuzzy"

## 4. Rollouts, termination, and policy evaluation

A rollout alternates model and tool steps. Terminate when the model emits a valid final SKU, when a non-retryable tool/grader error occurs, or when `max_episode_steps` is reached. The cap prevents loops and unbounded cost; it is not a target length. The grader evaluates the terminal answer, while separate diagnostics should track tool-called rate, argument validity, tool latency/errors, episode length, and exact-match reward.

Never reward merely calling a tool: the policy goal is correct tool use **and** a correct final answer.

In [ ]:
tool_binding_template = [{"name": "search_catalog", "server_url": "${FOUNDRY_TOOL_SERVER_URL}"}]
rft_method_template = {
    "type": "reinforcement",
    "reinforcement": {
        "grader": multi_grader,
        "tools": tool_binding_template,
        "max_episode_steps": 10,
        "hyperparameters": {
            "eval_interval": 3,
            "eval_samples": 5,
            "compute_multiplier": 1.0,
            "reasoning_effort": "medium",
        },
    },
}
job_plan = {
    "model": "${FOUNDRY_RFT_BASE_MODEL}",
    "training_file": str(TRAIN_PATH),
    "validation_file": str(VALIDATION_PATH),
    "suffix": "agentic-tool-calling",
    "method": rft_method_template,
}
print(json.dumps(job_plan, indent=2))


## 5. Upload/reuse, submit/reuse, and monitor to a terminal state

This cell performs real Foundry operations only when explicitly enabled. Uploads use meaningful content-addressed names and are reused by exact filename. `FOUNDRY_RFT_JOB_ID` may point to an existing job for monitoring without creating a new paid job. New submission additionally requires `FOUNDRY_RUN_PAID_JOBS=true`, the reviewed base model, and the HTTPS rollout tool URL. File and job waits are bounded and failed/cancelled terminal states raise with service details.


In [ ]:
existing_job_id = os.getenv("FOUNDRY_RFT_JOB_ID", "").strip()
terminal_rft_job = None
rft_job_reference = None

if RUN_PAID_JOBS or existing_job_id:
    config = load_foundry_config(ROOT / ".env" if (ROOT / ".env").is_file() else None)
    with create_project_context(config) as context:
        if existing_job_id:
            rft_job_reference = existing_job_id
        else:
            values = require_env("FOUNDRY_RFT_BASE_MODEL", "FOUNDRY_TOOL_SERVER_URL")
            train_upload = upload_or_reuse_file(
                context.openai_client,
                demo_slug="agentic-rft-tools",
                purpose_name="training",
                path=TRAIN_PATH,
                timeout_seconds=FILE_TIMEOUT_SECONDS,
            )
            validation_upload = upload_or_reuse_file(
                context.openai_client,
                demo_slug="agentic-rft-tools",
                purpose_name="validation",
                path=VALIDATION_PATH,
                timeout_seconds=FILE_TIMEOUT_SECONDS,
            )
            live_method = json.loads(json.dumps(rft_method_template))
            tool_binding = {
                "name": "search_catalog",
                "server_url": values["FOUNDRY_TOOL_SERVER_URL"],
            }
            tool_token = os.getenv("FOUNDRY_TOOL_AUTH_TOKEN", "").strip()
            if tool_token:
                tool_binding["headers"] = {"X-Functions-Key": tool_token}
            live_method["reinforcement"]["tools"] = [tool_binding]
            submitted = create_or_reuse_fine_tuning_job(
                context.openai_client,
                model=values["FOUNDRY_RFT_BASE_MODEL"],
                training_file_id=train_upload.resource_id,
                validation_file_id=validation_upload.resource_id,
                suffix="agentic-tool-calling",
                method=live_method,
            )
            rft_job_reference = submitted.resource_id
        terminal_rft_job = monitor_fine_tuning_job(
            context.openai_client,
            rft_job_reference,
            poll_interval_seconds=POLL_INTERVAL_SECONDS,
            timeout_seconds=JOB_TIMEOUT_SECONDS,
        )
        print({
            "job_status": terminal_rft_job.status,
            "fine_tuned_model_available": bool(getattr(terminal_rft_job, "fine_tuned_model", None)),
        })
else:
    print("No remote job operation. Set FOUNDRY_RFT_JOB_ID to monitor an existing job or FOUNDRY_RUN_PAID_JOBS=true to submit one.")


## 6. Resolve the caller-provided deployment, invoke it, and evaluate

`azure-ai-projects` 2.7 can discover deployments but does not create them through the same SDK surface. After a succeeded job, create a deployment through your approved management path and set `FOUNDRY_RFT_DEPLOYMENT`. The evaluation loop invokes that deployment through the Responses API, executes only the checked-in read-only `search_catalog` function locally, returns the tool output, and scores final SKU exact match on the preserved validation split. The loop has a fixed tool-step bound and raises on missing calls or missing final output.


In [ ]:
def predict_sku(context, deployment_name, row):
    response = invoke_response(
        context.openai_client,
        deployment_name=deployment_name,
        input=responses_input(row["messages"]),
        tools=responses_tools(row["tools"]),
        tool_choice="required",
    )
    for _ in range(4):
        calls = response_function_calls(response)
        if not calls:
            answer = response_output_text(response)
            if not answer:
                raise RuntimeError("Deployment returned neither a function call nor final text")
            return answer
        outputs = []
        for call in calls:
            if call["name"] != "search_catalog":
                raise RuntimeError(f"Unexpected function call: {call['name']}")
            arguments = json.loads(call["arguments"])
            result = execute_search_catalog(**arguments)
            outputs.append({
                "type": "function_call_output",
                "call_id": call["call_id"],
                "output": json.dumps(result),
            })
        response = invoke_response(
            context.openai_client,
            deployment_name=deployment_name,
            input=outputs,
            previous_response_id=response.id,
        )
    raise RuntimeError("Tool episode exceeded the four-step evaluation bound")

live_evaluation = None
if RUN_LIVE_EVALUATION:
    deployment_name = require_env("FOUNDRY_RFT_DEPLOYMENT")["FOUNDRY_RFT_DEPLOYMENT"]
    config = load_foundry_config(ROOT / ".env" if (ROOT / ".env").is_file() else None)
    with create_project_context(config) as context:
        deployment = ensure_deployment(
            context.project_client,
            deployment_name=deployment_name,
            timeout_seconds=600,
        )
        live_evaluation = evaluate_sku_answers(
            validation_rows[:EVALUATION_LIMIT],
            lambda row: predict_sku(context, deployment_name, row),
        )
        print({
            "deployment_resolved": getattr(deployment, "name", deployment_name),
            "exact": live_evaluation["exact"],
            "total": live_evaluation["total"],
        })
else:
    print("Live deployment evaluation skipped; set FOUNDRY_RUN_LIVE_EVALUATION=true after configuring FOUNDRY_RFT_DEPLOYMENT.")


## 7. Representative evaluation curve

The following curve is **illustrative**, not a claim about a completed job. It demonstrates why checkpoint selection should use held-out reward and failure diagnostics rather than the final training step alone.

![Illustrative Agentic RFT reward curves](../assets/charts/representative-reward-curve.svg)

The underlying values are preserved in [`representative-run.json`](../assets/metrics/representative-run.json).


In [ ]:
import matplotlib.pyplot as plt

steps = [0, 3, 6, 9, 12, 15]
training_reward = [0.32, 0.41, 0.50, 0.57, 0.63, 0.69]
validation_reward = [0.30, 0.39, 0.46, 0.54, 0.55, 0.53]
plt.plot(steps, training_reward, marker="o", label="illustrative train reward")
plt.plot(steps, validation_reward, marker="o", label="illustrative validation reward")
plt.xlabel("training step")
plt.ylabel("reward")
plt.ylim(0, 1)
plt.grid(alpha=0.25)
plt.legend();

## Conclusions and next curriculum step

You preserved split membership, validated both local and optional HTTPS tool contracts, calibrated a strict composite grader, and built a complete guarded RFT path: exact-file upload/reuse, submission or existing-job reuse, bounded terminal monitoring, explicit deployment resolution, real tool-using endpoint invocation, and exact-match evaluation. SFT would imitate demonstrated calls; Agentic RFT instead receives reward for the outcome of the bounded tool-using rollout.

**Next:** continue to `../02-retail-agent-capstone/notebooks/demo.ipynb`, where guarded SFT and RFT paths share policy, deployment, and held-out next-tool evaluation contracts.

*Variability footnote:* model outputs, reward trajectories, latency, and service behavior vary by model snapshot, sampling, region, preview API version, grader model, and remote tool reliability. Re-run held-out evaluation for your own configuration.

> **Variability note:** Actual results may vary by model version, data, configuration, region availability, and service conditions.
